# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors Exploration with `mlcroissant`
This notebook guides you in loading and exploring the FAIR^2 dataset package using the `mlcroissant` library.

### Dataset Source
The dataset is described by a [Croissant schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json) and documents the clinicopathological and molecular characteristics of second primary colorectal cancer in cancer survivors, including MSI-H status and anatomical distribution.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print human-readable metadata summary
meta = dataset.metadata
print(f"{meta.name}: {meta.description}")
print(f"\nDataset identifier: {meta.identifier}")
print(f"Version: {meta.version}")
print(f"License: {meta.license}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s.

We'll enumerate all record sets in the dataset, inspecting their `@id`s and the columns (fields) they provide.

In [ ]:
# List all record sets along with their fields and IDs
record_sets_info = []
for record_set in dataset.record_sets():
    rs_id = record_set['@id']
    rs_name = record_set.get('name', rs_id)
    columns = record_set.get('field', [])
    if not isinstance(columns, list):
        columns = [columns]
    fields_info = []
    for fld in columns:
        # Each field can be {'@id': ...} or a string
        field_id = fld['@id'] if isinstance(fld, dict) and '@id' in fld else str(fld)
        fields_info.append(field_id)
    record_sets_info.append({'@id': rs_id, 'name': rs_name, 'fields': fields_info})

for rs in record_sets_info:
    print(f"RecordSet: {rs['name']} (@id: {rs['@id']})")
    print("  Fields/Columns (by @id):")
    for fid in rs['fields']:
        print(f"    - {fid}")
    print()

## 3. Data Extraction

Load data from available record sets into pandas DataFrames. All references are made using the `@id`.
We'll load each record set one by one and preview their columns and first records.

In [ ]:
# Extract data from each record set using mlcroissant. Use record set @id references.
dataframes = {}
# Store the full list of record_set_ids
record_set_ids = [rs['@id'] for rs in record_sets_info]

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    dataframes[record_set_id] = pd.DataFrame(records)

# Show DataFrame info for each RecordSet
for record_set_id in record_set_ids:
    df = dataframes[record_set_id]
    print(f"=== RecordSet @id: {record_set_id} ===")
    print(f"Columns: {df.columns.tolist()}")
    display(df.head())
    print()
# For further analysis, select the primary tabular record set (the first one here for demonstration)
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id:
    print(f"Primary data RecordSet selected: {main_record_set_id}")

## 4. Exploratory Data Analysis (EDA)

Let’s explore and process the main clinical record set. For this, identify a plausible numeric column (such as patient age or diagnosis interval), filter, normalize, and group by categorical attributes, always referencing fields by their `@id`.

In [ ]:
# --- Example EDA Steps ---
# Adjust the field IDs below according to the data overview section above. Replace these strings as appropriate.

# Configuration (update these to match available columns):
record_set_id = main_record_set_id
# e.g. '@id' for a numeric field, such as age, or interval between cancers
numeric_field_id = None
# e.g. '@id' for a group/categorical field, such as sex or MSI status
group_field_id = None

# Attempt to suggest potential numeric and group fields from the DataFrame
if record_set_id:
    df = dataframes[record_set_id]
    print("Available columns:", df.columns.tolist())

    # Heuristics: find a likely numeric column (contains 'interval', 'age', or is of int/float type)
    for col in df.columns:
        if any(word in col.lower() for word in ['interval', 'age', 'duration', 'years']) and pd.api.types.is_numeric_dtype(df[col]):
            numeric_field_id = col
            break
    else:
        # fallback: pick any numeric column
        num_cols = df.select_dtypes(include=['number']).columns
        if len(num_cols)>0:
            numeric_field_id = num_cols[0]

    # Suggest a grouping column (e.g. 'sex', 'msi', 'site', or any object column)
    for col in df.columns:
        if any(word in col.lower() for word in ['sex', 'msi', 'site', 'group', 'status']):
            group_field_id = col
            break
    else:
        obj_cols = df.select_dtypes(include=['object']).columns
        group_field_id = obj_cols[0] if len(obj_cols)>0 else None

    print(f"Selected numeric field (by @id): {numeric_field_id}")
    print(f"Selected group field (by @id): {group_field_id}")

    # Only proceed if we have a valid numeric field
    if numeric_field_id and df[numeric_field_id].dropna().size > 0:
        # Remove outliers using a simple threshold if possible
        try:
            threshold = df[numeric_field_id].quantile(0.95)
            filtered_df = df[df[numeric_field_id] <= threshold]
        except Exception:
            filtered_df = df.copy()

        print(f"Filtered records (below 95th percentile) for {numeric_field_id} (threshold={threshold if 'threshold' in locals() else 'NA'}):")
        display(filtered_df.head())

        # Normalize the numeric field (z-score)
        filtered_df[f"{numeric_field_id}_normalized"] = (
            (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
        )
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by the group field if available
        if group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean()
            print(f"\nAverage {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
    else:
        print("No suitable numeric field identified for EDA.")
else:
    print("No primary record set available.")

## 5. Visualization

Now visualize the distribution of the selected numeric variable and means by the grouping variable. All axes and legends are labeled with the used `@id`s.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualize numeric field distribution and group means
if record_set_id and numeric_field_id and numeric_field_id in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_field_id], kde=True)
    plt.title(f"Distribution of {numeric_field_id} (@id)")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        sns.boxplot(x=group_field_id, y=numeric_field_id, data=df)
        plt.title(f"{numeric_field_id} by {group_field_id} (@id)")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=30, ha='right')
        plt.show()
else:
    print("Not enough data for numeric field visualization.")

## 6. Conclusion

- This notebook demonstrated FAIR loading and processing of the clinical dataset on second primary colorectal cancer using the `mlcroissant` library, with all references by record set and field `@id`.
- The workflow included structured extraction, filtering, normalization, grouping and visualization steps, all referencing schema `@id`s for reproducibility and semantic clarity.

**Next Steps:**
- You may now extend this notebook to advanced statistical analyses or machine learning workflows, always referencing fields by their `@id` for interoperability and transparency.